# B3 · Skewed variables: log, Yeo-Johnson and quantiles, inside and outside the range

**Type**: Implement with scikit-learn · **Level**: ●●○ independent · **Time**: about 40 minutes · **core path ★**

## Objective

Compare three transformations of skewed variables (log, `PowerTransformer`, `QuantileTransformer`) on the skewness and
on the performance of a linear model, and verify that the `QuantileTransformer` **cannot extrapolate**.

## Dataset

Synthetic: 1200 rows, two very skewed features `x1` and `x2` (lognormal) and a normal one `x3`, with
`y = 2·log(x1) − 1.5·log(x2) + x3 + noise`. The linear model on the raw features is therefore wrong, and on log(x) it is correct.
`X_tr`/`X_te` (random 50/50 split) and `X_extreme`: the same points as `X_te` with `x1` and `x2` multiplied by 20, that is
**outside the range** seen in training.

## Educational purpose

Understand that the transformations are not interchangeable: the quantile transformer produces perfect distributions but
**saturates outside the range of the data**, while a continuous function like the log keeps working (and Yeo-Johnson degrades gradually).

## Task

1. Compute `skewness`, a DataFrame with `x1` and `x2` as rows and as columns `raw`, `log`, `yeo_johnson`, `quantile` (`scipy.stats.skew` on the training set after each transformation; transformers fitted on the training set).
2. For each transformation build a pipeline (`FunctionTransformer(np.log)` on x1 and x2 only: use `ColumnTransformer`) with a final `Ridge(alpha=1)`, train it on `X_tr` and save in `r2_normal` and `r2_extreme` the R² on the normal test set and on the out-of-range test set.
3. Assign to `best_normal` and `best_extreme` the name of the transformation with the highest R² in the two cases.
4. Explain in `answer` ("a", "b" or "c") why the `QuantileTransformer` does badly out of range:
   - **a**: the quantiles are computed on the training set: any value beyond the training maximum gets the same value (saturation), so the model does not see the difference between 3 and 20 times the maximum;
   - **b**: the QuantileTransformer returns non-finite values out of range;
   - **c**: Ridge does not work with Gaussian data.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, PowerTransformer, QuantileTransformer

rng = np.random.default_rng(0)
n = 1200
X = pd.DataFrame({"x1": rng.lognormal(0, 1, n), "x2": rng.lognormal(1, 0.8, n), "x3": rng.normal(size=n)})
y = 2 * np.log(X.x1) - 1.5 * np.log(X.x2) + X.x3 + rng.normal(0, 0.5, n)
X_tr, X_te = X.iloc[:600], X.iloc[600:]
y_tr, y_te = y.iloc[:600], y.iloc[600:]
X_extreme = X_te.copy()
X_extreme[["x1", "x2"]] *= 20                                   # outside the training range
y_extreme = 2 * np.log(X_extreme.x1) - 1.5 * np.log(X_extreme.x2) + X_extreme.x3 + (y_te - (2 * np.log(X_te.x1) - 1.5 * np.log(X_te.x2) + X_te.x3))
COL = ["x1", "x2"]

## Your work

In [ ]:
transforms = {
    "raw": "passthrough",
    "log": FunctionTransformer(np.log),
    "yeo_johnson": PowerTransformer(),
    "quantile": QuantileTransformer(n_quantiles=200, output_distribution="normal"),
}
skewness = pd.DataFrame(index=COL, columns=list(transforms), dtype=float)
for name, t in transforms.items():
    ...
skewness.round(2)

In [ ]:
r2_normal, r2_extreme = {}, {}
for name, t in transforms.items():
    model = ...
    ...
best_normal = ...
best_extreme = ...

In [ ]:
answer = ...

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='Pipeline: `make_pipeline(ColumnTransformer([("t", t, COL)], remainder="passthrough"), Ridge(alpha=1))`; for `"passthrough"` it works the same (`ColumnTransformer` accepts the string). Skewness: `stats.skew(t.fit_transform(X_tr[[c]]).ravel())`; for `"passthrough"` use `X_tr[c]`.'):
    check(list(skewness.columns) == ["raw", "log", "yeo_johnson", "quantile"], "correct columns", "raw, log, yeo_johnson, quantile")
    check((skewness["raw"] > 2).all() and (skewness.drop(columns="raw").abs() < 0.3).all().all(),
          "the three transformations reduce the skewness (from above 2 to below 0.3 in absolute value)", "the skewness of the transformed ones must be close to zero")
    check(r2_normal["log"] > r2_normal["raw"] + 0.2, "the log improves the linear model a lot compared to the raw",
          "r2 with the log must exceed the raw one by more than 0.2")
    check(abs(r2_normal["yeo_johnson"] - r2_normal["log"]) < 0.05, "Yeo-Johnson is equivalent to the log on the normal data", "on the data inside the range they must almost coincide")
    check(r2_extreme["log"] > r2_extreme["quantile"] + 0.3, "out of range the log stays good and the quantiles collapse",
          "on the extreme data the quantile transformer should be much worse than the log")
    check(best_extreme in ("log", "yeo_johnson") and best_extreme != "quantile", "the best out of range is not the quantile transformer", "look at r2_extreme")
    check(answer == "a", "correct explanation", "reread: what does QuantileTransformer do with a value beyond the maximum seen?")
done()

---
The commented solution is in `solutions/B_with_sklearn/B3_transformations_of_skewed_variables.ipynb`. Open it after you pass the check, or after you have really tried.